In [ ]:
print("hi")

# Sampling-parameter validation: Chronos-T5 Base (200M)

Does changing Chronos's own sampling settings reduce overconfidence and improve
M6 RPS, without any post-hoc probability shrinkage? The weights stay frozen.
Only inference-time sampling changes.

| | |
|---|---|
| Model | `amazon/chronos-t5-base`, zero-shot, frozen |
| Variables | `temperature` x `top_k` (18 combinations) |
| Fixed | context 512 weekdays, horizon 20, 500 samples, seed 42, official M6 asset order |
| Input | the 6 shared pre-M6 validation contexts on Drive |
| Output | 108 NPZ files (6 origins x 18 configurations) |

The six validation origins are weekly Fridays from 2021-12-31 to 2022-02-04.
Their 20-weekday targets all finish on or before 2022-03-04, before the first
scored M6 target on 2022-03-07, so the 12 M6 rounds stay untouched while a
sampling configuration is chosen.

This notebook only generates and saves raw forecasts. It computes no RPS, ECE,
confidence or entropy, and applies no shrinkage or other calibration. Scoring
happens later with the project's existing local code.

Each configuration draws 500 trajectories as five consecutive 100-sample chunks,
with the seed set once at the start of that configuration. Contexts, seed,
chunking and batch size are identical across configurations, so a difference
between two of them reflects the sampling settings rather than a different draw.

Expect roughly 4 minutes per configuration, so about 7 hours for all 108. That
is longer than one Colab session, which is why every file is written and
verified as it is produced and why re-running skips finished work. Plan to run
this over several sessions, and run only one model per session.

## 1. Install Chronos

In [ ]:
%pip install -q chronos-forecasting

## 2. Imports and settings

The sampling grid is the only thing that differs from the established runs.
Everything else, including the context length, horizon, seed, batch size,
chunking and asset order, is carried over unchanged so the comparison is
controlled.

`top_k = "full_vocabulary"` is resolved from the loaded model's own
configuration in section 7, never hard-coded.

In [ ]:
import json
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from inspect import signature
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import ChronosPipeline

# --- Model (frozen, inference only) ----------------------------------------
MODEL_ID = "amazon/chronos-t5-base"

# --- The experimental variables --------------------------------------------
TEMPERATURES = [1.0, 1.5, 2.0, 3.0, 3.5, 5.0]
TOP_K_VALUES = [50, 500, "full_vocabulary"]
TOP_P = 1.0

# --- Everything else is fixed, exactly as in the established runs -----------
NUM_SAMPLES = 500
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
RANDOM_SEED = 42
SERIES_BATCH_SIZE = 10                # lower ONLY on CUDA OOM

# 500 samples in one predict() call exhausts the GPU, so each configuration is
# drawn in chunks of the request size already known to work at batch size 10.
SAMPLE_CHUNK_SIZE = 100
N_CHUNKS, remainder = divmod(NUM_SAMPLES, SAMPLE_CHUNK_SIZE)
assert remainder == 0, "NUM_SAMPLES must be a whole number of chunks"

VALIDATION_IDS = range(1, 7)

# id -> context start, origin (= context end), target start, target end
# Taken from Data/validation_contexts/validation_context_manifest.csv and
# cross-checked against the manifest on Drive in section 5.
VALIDATION_SCHEDULE = {
    1: ("2020-01-16", "2021-12-31", "2022-01-03", "2022-01-28"),
    2: ("2020-01-23", "2022-01-07", "2022-01-10", "2022-02-04"),
    3: ("2020-01-30", "2022-01-14", "2022-01-17", "2022-02-11"),
    4: ("2020-02-06", "2022-01-21", "2022-01-24", "2022-02-18"),
    5: ("2020-02-13", "2022-01-28", "2022-01-31", "2022-02-25"),
    6: ("2020-02-20", "2022-02-04", "2022-02-07", "2022-03-04"),
}

# Genuine leading missing history: CARR and OGN listed late, so their early
# returns do not exist. These values must be left as NaN and handled by the
# model's missing-value mask, never filled or dropped.
EXPECTED_LEADING_NAN = {
    1: {"CARR": 46, "OGN": 347},
    2: {"CARR": 41, "OGN": 342},
    3: {"CARR": 36, "OGN": 337},
    4: {"CARR": 31, "OGN": 332},
    5: {"CARR": 26, "OGN": 327},
    6: {"CARR": 21, "OGN": 322},
}

# Official M6 asset order - used to verify the contexts, not to reorder them.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

# The installed Chronos must accept the sampling arguments, or the grid would
# silently collapse to the model defaults.
_predict_params = signature(ChronosPipeline.predict).parameters
_missing = [p for p in ("temperature", "top_k", "top_p") if p not in _predict_params]
assert not _missing, (
    f"This chronos-forecasting build does not accept {_missing} in predict(); "
    "the sampling grid cannot be applied."
)

CONFIGURATIONS = [(t, k) for t in TEMPERATURES for k in TOP_K_VALUES]
TOTAL_RUNS = len(CONFIGURATIONS) * len(list(VALIDATION_IDS))

print(f"chronos-forecasting {version('chronos-forecasting')} | torch {torch.__version__}")
print(f"Experiment: {MODEL_ID}")
print(f"Grid: {len(TEMPERATURES)} temperatures x {len(TOP_K_VALUES)} top-k "
      f"= {len(CONFIGURATIONS)} configurations")
print(f"      x {len(list(VALIDATION_IDS))} validation origins = {TOTAL_RUNS} saved files")
print(f"Fixed: {NUM_SAMPLES} samples in {N_CHUNKS} chunks of {SAMPLE_CHUNK_SIZE}, "
      f"context {CONTEXT_LENGTH}, horizon {PREDICTION_LENGTH}, seed {RANDOM_SEED}, "
      f"top_p {TOP_P}, {N_ASSETS} assets")
print("predict() accepts temperature, top_k and top_p")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configure paths

Validation contexts are read from the shared folder that both models use. The
only path specific to this notebook is the output base; one subfolder per
configuration is created automatically, so no two configurations can overwrite
each other.

In [ ]:
# Shared validation contexts - the same six files for both models.
DRIVE_VALIDATION_CONTEXT_DIR = Path(
    "/content/drive/MyDrive/HonoursResearch/validation_contexts")

# >>> THE ONLY NEW PATH TO CONFIGURE <<<
DRIVE_OUTPUT_BASE_DIR = Path(
    "/content/drive/MyDrive/HonoursResearch/outputs/chronos_t5_base"
    "/temperature_validation")

OUTPUT_PREFIX = "chronos_t5_base"
MANIFEST_NAME = "validation_context_manifest.csv"


def context_path(validation_id: int) -> Path:
    return DRIVE_VALIDATION_CONTEXT_DIR / f"validation_{validation_id:02d}_context.csv"


def top_k_label(top_k) -> str:
    """Short, stable tag for a top-k setting, used in folder and file names."""
    return "full" if top_k == "full_vocabulary" else str(top_k)


def config_dir(temperature: float, top_k) -> Path:
    return DRIVE_OUTPUT_BASE_DIR / f"temp{temperature:.1f}_topk{top_k_label(top_k)}"


def samples_path(validation_id: int, temperature: float, top_k) -> Path:
    return (config_dir(temperature, top_k)
            / f"{OUTPUT_PREFIX}_validation{validation_id:02d}"
              f"_temp{temperature:.1f}_topk{top_k_label(top_k)}"
              f"_samples{NUM_SAMPLES}.npz")


if not DRIVE_VALIDATION_CONTEXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Validation context folder not found:\n  {DRIVE_VALIDATION_CONTEXT_DIR}\n"
        "Copy the six validation_NN_context.csv files and "
        f"{MANIFEST_NAME} there from Data/validation_contexts/."
    )

for temperature, top_k in CONFIGURATIONS:
    config_dir(temperature, top_k).mkdir(parents=True, exist_ok=True)

print(f"Validation contexts (shared): {DRIVE_VALIDATION_CONTEXT_DIR}")
print(f"Output base (new)           : {DRIVE_OUTPUT_BASE_DIR}")
print(f"Configuration folders       : {len(CONFIGURATIONS)}")
print(f"\nExample folder  : {config_dir(*CONFIGURATIONS[0]).name}")
print(f"Example filename: {samples_path(1, *CONFIGURATIONS[0]).name}")

## 5. Check the validation contexts

All six files plus the manifest must be present, and the dates hard-coded above
must agree with the manifest written by the repository pipeline. A mismatch here
means the wrong files were copied to Drive.

In [ ]:
missing = [context_path(v).name for v in VALIDATION_IDS
           if not context_path(v).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing validation contexts in {DRIVE_VALIDATION_CONTEXT_DIR}:\n  "
        + "\n  ".join(missing))

manifest_path = DRIVE_VALIDATION_CONTEXT_DIR / MANIFEST_NAME
if not manifest_path.is_file():
    raise FileNotFoundError(f"Missing {MANIFEST_NAME} in {DRIVE_VALIDATION_CONTEXT_DIR}")

# Cross-check this notebook's schedule against the pipeline's own manifest.
manifest = pd.read_csv(manifest_path).set_index("validation_id")
for v in VALIDATION_IDS:
    ctx_start, origin, target_start, target_end = VALIDATION_SCHEDULE[v]
    row = manifest.loc[v]
    expected = (row["context_start"], row["context_end"],
                row["target_start"], row["target_end"])
    assert (ctx_start, origin, target_start, target_end) == expected, (
        f"Validation {v}: notebook schedule {(ctx_start, origin, target_start, target_end)} "
        f"does not match the manifest {expected}")
    assert int(row["n_assets"]) == N_ASSETS, f"Validation {v}: manifest asset count"
    assert int(row["context_rows"]) == CONTEXT_LENGTH, f"Validation {v}: manifest row count"

print(f"All {len(list(VALIDATION_IDS))} validation contexts present in "
      f"{DRIVE_VALIDATION_CONTEXT_DIR}")
print("Notebook schedule agrees with the pipeline manifest\n")
for v in VALIDATION_IDS:
    ctx_start, origin, target_start, target_end = VALIDATION_SCHEDULE[v]
    print(f"  {v}: context {ctx_start} .. {origin} | target {target_start} .. {target_end}")

## 6. GPU check

A GPU is required. This cell stops rather than silently running on CPU.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Runtime > Change runtime type > Hardware "
        "accelerator: GPU, then re-run from section 3."
    )

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
GPU_NAME = torch.cuda.get_device_name(0)

print(f"GPU   : {GPU_NAME}")
print(f"dtype : {DTYPE}")

## 7. Load the model, once, and resolve the full vocabulary

Same checkpoint, CUDA and dtype setup as the established runs. Loaded here and
reused for every configuration, never reloaded inside the loop. Inference only:
no training, fine-tuning or weight modification.

`top_k = "full_vocabulary"` means "do not truncate the token distribution", so
it resolves to this checkpoint's own vocabulary size, read from the loaded
model's Chronos configuration. The model's default sampling settings are printed
too, because one grid point should coincide with them.

In [ ]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

pipeline = ChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map="cuda",
    torch_dtype=DTYPE,
)

n_params = sum(p.numel() for p in pipeline.model.parameters())
MODEL_LOADED_ONCE = True


def resolve_full_vocabulary_top_k(loaded) -> int:
    """This checkpoint's token-vocabulary size, for untruncated sampling.

    Read from the loaded model rather than assumed, because the value differs
    between Chronos checkpoints. Two known layouts are tried before giving up.
    """
    chronos_config = getattr(loaded.model, "config", None)
    n_tokens = getattr(chronos_config, "n_tokens", None)
    if n_tokens is None:
        raw_config = getattr(getattr(loaded.model, "model", None), "config", None)
        raw_chronos = getattr(raw_config, "chronos_config", None)
        if isinstance(raw_chronos, dict):
            n_tokens = raw_chronos.get("n_tokens")
    if n_tokens is None:
        raise RuntimeError(
            "Could not read n_tokens from the loaded model. Inspect "
            "pipeline.model.config and set the full-vocabulary top-k manually "
            "rather than guessing a value."
        )
    return int(n_tokens)


FULL_VOCABULARY_TOP_K = resolve_full_vocabulary_top_k(pipeline)


def resolved_top_k(top_k) -> int:
    """Map a grid entry to the integer top-k actually passed to predict()."""
    return FULL_VOCABULARY_TOP_K if top_k == "full_vocabulary" else int(top_k)


_defaults = getattr(pipeline.model, "config", None)
print(f"Loaded: {MODEL_ID}")
print(f"  parameters      : {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"  device          : {GPU_NAME} | dtype {DTYPE}")
print(f"  vocabulary size : {FULL_VOCABULARY_TOP_K} "
      f"-> this is the resolved 'full_vocabulary' top-k")
print(f"  model defaults  : temperature={getattr(_defaults, 'temperature', 'unknown')}, "
      f"top_k={getattr(_defaults, 'top_k', 'unknown')}, "
      f"top_p={getattr(_defaults, 'top_p', 'unknown')}")
print("Model loaded ONCE - reused for every configuration and every origin.")

## 8. Helpers

`load_and_validate_context` reads one validation context and checks its shape,
dates, asset order and missing-value pattern. `generate_samples` draws 500
trajectories in chunks for one configuration, with the seed set once at the
start so each configuration is independently reproducible.
`save_and_verify` writes the NPZ with its run metadata and reads it back.

The raw array keys are the same three the established notebooks write, so the
project's existing scoring code can read these files unchanged. Run metadata is
stored alongside them as extra keys.

Aggressive settings are expected to produce wide forecasts. Values are recorded
and reported exactly as generated: nothing is clipped, filled or rescaled. A
non-finite value does not abort the run, it is saved and flagged, because
observing that behaviour is part of the experiment.

In [ ]:
def load_and_validate_context(validation_id: int):
    """Load one validation context, validate it, and build the model input."""
    ctx_start, origin, target_start, target_end = VALIDATION_SCHEDULE[validation_id]
    df = pd.read_csv(context_path(validation_id), parse_dates=["date"])

    assert df.shape[0] == CONTEXT_LENGTH, (
        f"Validation {validation_id}: expected {CONTEXT_LENGTH} rows, found {df.shape[0]}"
    )
    assert list(df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
        f"Validation {validation_id}: columns are not 'date' + the official M6 asset order"
    )
    dates = df["date"]
    assert dates.is_monotonic_increasing, f"Validation {validation_id}: dates not ascending"
    assert not dates.duplicated().any(), f"Validation {validation_id}: duplicate dates"
    assert dates.iloc[0] == pd.Timestamp(ctx_start), f"Validation {validation_id}: context start"
    assert dates.iloc[-1] == pd.Timestamp(origin), f"Validation {validation_id}: context end/origin"
    assert (dates <= pd.Timestamp(origin)).all(), (
        f"Validation {validation_id}: context contains a date after the origin"
    )

    matrix = df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
    assert matrix.shape == (N_ASSETS, CONTEXT_LENGTH)
    assert np.array_equal(
        matrix, df[OFFICIAL_ASSET_ORDER].to_numpy().T, equal_nan=True
    ), f"Validation {validation_id}: model input differs from the loaded context values"

    # Leading gaps are genuine later inceptions and stay NaN; anything after an
    # asset's first real value would mean the wrong file was copied.
    leading_missing, interior_missing = {}, {}
    for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
        row = matrix[i]
        n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
        if n_lead:
            leading_missing[symbol] = n_lead
        if np.isnan(row[n_lead:]).any():
            interior_missing[symbol] = int(np.isnan(row[n_lead:]).sum())
    assert not interior_missing, (
        f"Validation {validation_id}: unexpected interior NaNs {interior_missing}"
    )
    assert leading_missing == EXPECTED_LEADING_NAN[validation_id], (
        f"Validation {validation_id}: leading NaN pattern changed - "
        f"expected {EXPECTED_LEADING_NAN[validation_id]}, found {leading_missing}"
    )

    forecast_dates = pd.bdate_range(target_start, target_end)
    assert len(forecast_dates) == PREDICTION_LENGTH, (
        f"Validation {validation_id}: target window is not {PREDICTION_LENGTH} weekdays"
    )
    return matrix, forecast_dates


def generate_samples(context_matrix, validation_id: int, temperature: float, top_k):
    """Draw NUM_SAMPLES trajectories for one configuration; (100, 500, 20).

    The seed is set ONCE here and deliberately not reset between chunks:
    re-seeding would make every chunk an identical copy of the first. Because
    the seed, context, chunking and batch size are the same for every
    configuration, differences between configurations come from the sampling
    settings rather than from a different random draw.
    """
    k = resolved_top_k(top_k)

    torch.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)

    chunks = []
    for chunk in range(N_CHUNKS):
        batch_outputs = []
        for start in range(0, N_ASSETS, SERIES_BATCH_SIZE):
            stop = min(start + SERIES_BATCH_SIZE, N_ASSETS)
            batch_context = [
                torch.tensor(context_matrix[i], dtype=torch.float32)
                for i in range(start, stop)
            ]
            samples = pipeline.predict(
                inputs=batch_context,
                prediction_length=PREDICTION_LENGTH,
                num_samples=SAMPLE_CHUNK_SIZE,
                temperature=temperature,
                top_k=k,
                top_p=TOP_P,
            )
            batch_outputs.append(samples.to(torch.float32).cpu().numpy())
            # Release the GPU tensor as soon as it is on the host.
            del samples, batch_context
        torch.cuda.empty_cache()

        chunk_samples = np.concatenate(batch_outputs, axis=0)
        assert chunk_samples.shape == (N_ASSETS, SAMPLE_CHUNK_SIZE, PREDICTION_LENGTH), (
            f"Validation {validation_id} chunk {chunk + 1}: got {chunk_samples.shape}"
        )
        chunks.append(chunk_samples)

    forecast_samples = np.concatenate(chunks, axis=1)
    assert forecast_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH), (
        f"Validation {validation_id}: expected "
        f"({N_ASSETS}, {NUM_SAMPLES}, {PREDICTION_LENGTH}), got {forecast_samples.shape}"
    )
    # Identical chunks would mean the seed was reset mid-configuration.
    assert not np.array_equal(
        forecast_samples[:, :SAMPLE_CHUNK_SIZE],
        forecast_samples[:, SAMPLE_CHUNK_SIZE:2 * SAMPLE_CHUNK_SIZE]
    ), f"Validation {validation_id}: chunk 2 repeats chunk 1 - the seed was reset"
    return forecast_samples


def describe_values(forecast_samples) -> dict:
    """Range and finiteness of a raw forecast array, for reporting only."""
    finite = np.isfinite(forecast_samples)
    n_nonfinite = int((~finite).sum())
    values = forecast_samples[finite]
    return {
        "all_finite": n_nonfinite == 0,
        "n_nonfinite": n_nonfinite,
        "min": float(values.min()) if values.size else float("nan"),
        "max": float(values.max()) if values.size else float("nan"),
        "abs_max": float(np.abs(values).max()) if values.size else float("nan"),
    }


def save_and_verify(validation_id: int, temperature: float, top_k,
                    forecast_samples, forecast_dates) -> Path:
    """Save the raw array plus run metadata, then reload and check it."""
    path = samples_path(validation_id, temperature, top_k)
    ctx_start, origin, target_start, target_end = VALIDATION_SCHEDULE[validation_id]
    date_strings = np.array([d.strftime("%Y-%m-%d") for d in forecast_dates])

    metadata = {
        "model_id": MODEL_ID,
        "validation_id": validation_id,
        "forecast_origin": origin,
        "context_start": ctx_start,
        "target_start": target_start,
        "target_end": target_end,
        "temperature": temperature,
        "top_k_setting": str(top_k),
        "resolved_top_k": resolved_top_k(top_k),
        "top_p": TOP_P,
        "num_samples": NUM_SAMPLES,
        "prediction_length": PREDICTION_LENGTH,
        "random_seed": RANDOM_SEED,
        "sample_chunk_size": SAMPLE_CHUNK_SIZE,
        "series_batch_size": SERIES_BATCH_SIZE,
        "context_length": CONTEXT_LENGTH,
        "dtype": str(DTYPE),
        "generated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        **describe_values(forecast_samples),
    }

    # The first three keys match the established NPZ format, so existing
    # scoring code reads these files unchanged; run_metadata is extra.
    np.savez_compressed(
        path,
        forecast_samples=forecast_samples,
        asset_symbols=np.array(OFFICIAL_ASSET_ORDER),
        forecast_dates=date_strings,
        run_metadata=np.array(json.dumps(metadata)),
    )

    with np.load(path, allow_pickle=False) as reloaded:
        r_samples = reloaded["forecast_samples"]
        r_symbols = list(reloaded["asset_symbols"])
        r_dates = list(reloaded["forecast_dates"])
        r_meta = json.loads(str(reloaded["run_metadata"]))

    assert r_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH), "reloaded shape"
    assert r_symbols == OFFICIAL_ASSET_ORDER, "asset ordering changed on save/reload"
    assert np.array_equal(r_samples, forecast_samples, equal_nan=True), "saved values differ"
    assert r_dates == list(date_strings), "forecast dates changed on save/reload"
    assert r_meta["temperature"] == temperature, "metadata temperature"
    assert r_meta["resolved_top_k"] == resolved_top_k(top_k), "metadata top-k"
    return path


def already_done(validation_id: int, temperature: float, top_k) -> bool:
    """True if this configuration is already saved and passes its checks.

    Used for resume: a run interrupted by a Colab disconnect can be restarted
    and will skip everything already finished. A file that fails any check is
    treated as not done and is regenerated.
    """
    path = samples_path(validation_id, temperature, top_k)
    if not path.is_file():
        return False
    ctx_start, origin, target_start, target_end = VALIDATION_SCHEDULE[validation_id]
    expected_dates = [d.strftime("%Y-%m-%d")
                      for d in pd.bdate_range(target_start, target_end)]
    try:
        with np.load(path, allow_pickle=False) as f:
            if not {"forecast_samples", "asset_symbols",
                    "forecast_dates", "run_metadata"} <= set(f.files):
                return False
            if f["forecast_samples"].shape != (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH):
                return False
            if list(f["asset_symbols"]) != OFFICIAL_ASSET_ORDER:
                return False
            if list(f["forecast_dates"]) != expected_dates:
                return False
            meta = json.loads(str(f["run_metadata"]))
        return (meta.get("model_id") == MODEL_ID
                and meta.get("temperature") == temperature
                and meta.get("resolved_top_k") == resolved_top_k(top_k)
                and meta.get("top_p") == TOP_P
                and meta.get("random_seed") == RANDOM_SEED)
    except Exception:
        # An unreadable or truncated file counts as not done.
        return False


print("Helpers defined.")

### Run the validation grid

One pass over the six origins and the 18 configurations. Each finished file is
written and verified before the next begins, so a disconnect costs only the
configuration in progress. Re-running this cell resumes: anything already saved
and passing its checks is skipped.

In [ ]:
records, failures, skipped = [], [], []
experiment_started = datetime.now(timezone.utc)

for v in VALIDATION_IDS:
    ctx_start, origin, target_start, target_end = VALIDATION_SCHEDULE[v]
    print(f"\n{'=' * 74}\nValidation {v} | origin {origin} | "
          f"target {target_start} .. {target_end}\n{'=' * 74}")

    # Load and validate the context once, then reuse it for all 18 settings.
    try:
        context_matrix, forecast_dates = load_and_validate_context(v)
    except Exception as exc:
        failures.append((v, None, None, f"{type(exc).__name__}: {exc}"))
        print(f"  !! CONTEXT {v} FAILED - skipping all its configurations")
        traceback.print_exc()
        continue

    for temperature, top_k in CONFIGURATIONS:
        tag = f"temp {temperature:.1f} | top_k {top_k_label(top_k)}"
        if already_done(v, temperature, top_k):
            skipped.append((v, temperature, top_k))
            print(f"  {tag:<28} already saved and verified, skipped")
            continue

        try:
            started = datetime.now(timezone.utc)
            forecast_samples = generate_samples(context_matrix, v, temperature, top_k)
            elapsed = (datetime.now(timezone.utc) - started).total_seconds()

            stats = describe_values(forecast_samples)
            saved_path = save_and_verify(v, temperature, top_k,
                                         forecast_samples, forecast_dates)
            records.append({
                "validation_id": v, "origin": origin,
                "temperature": temperature, "top_k": top_k_label(top_k),
                "resolved_top_k": resolved_top_k(top_k),
                "seconds": round(elapsed, 1), "file": saved_path.name, **stats,
            })
            flag = "" if stats["all_finite"] else f"  !! {stats['n_nonfinite']} NON-FINITE"
            print(f"  {tag:<28} {elapsed:6.1f}s  range "
                  f"[{stats['min']:+.4f}, {stats['max']:+.4f}]  saved{flag}")

            del forecast_samples
            torch.cuda.empty_cache()

        except Exception as exc:
            failures.append((v, temperature, top_k, f"{type(exc).__name__}: {exc}"))
            print(f"  !! {tag} FAILED - finished configurations remain saved")
            traceback.print_exc()
            torch.cuda.empty_cache()

    del context_matrix
    torch.cuda.empty_cache()

total_minutes = (datetime.now(timezone.utc) - experiment_started).total_seconds() / 60
print(f"\nFinished in {total_minutes:.1f} min. "
      f"Generated {len(records)}, skipped {len(skipped)}, failed {len(failures)} "
      f"of {TOTAL_RUNS} configurations.")
if failures:
    print("FAILED:", [(v, t, top_k_label(k)) for v, t, k, _ in failures])

## 9. Verify every saved file

Re-reads every file on Drive, independently of the loop above, and checks the
array shape, asset order, forecast dates and stored metadata. Value ranges are
reported rather than judged: a wide range at a high temperature is a result, not
an error.

In [ ]:
ALL_VERIFIED = True
NONFINITE_FILES = []
verified_rows = []

for temperature, top_k in CONFIGURATIONS:
    for v in VALIDATION_IDS:
        path = samples_path(v, temperature, top_k)
        if not path.is_file():
            ALL_VERIFIED = False
            verified_rows.append((v, temperature, top_k_label(top_k), "MISSING", None, None))
            continue

        _, _, target_start, target_end = VALIDATION_SCHEDULE[v]
        expected_dates = [d.strftime("%Y-%m-%d")
                          for d in pd.bdate_range(target_start, target_end)]
        with np.load(path, allow_pickle=False) as f:
            samples = f["forecast_samples"]
            symbols = list(f["asset_symbols"])
            fdates = list(f["forecast_dates"])
            meta = json.loads(str(f["run_metadata"]))

        stats = describe_values(samples)
        ok = (samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH)
              and symbols == OFFICIAL_ASSET_ORDER
              and fdates == expected_dates
              and meta["model_id"] == MODEL_ID
              and meta["temperature"] == temperature
              and meta["resolved_top_k"] == resolved_top_k(top_k)
              and meta["top_p"] == TOP_P
              and meta["num_samples"] == NUM_SAMPLES
              and meta["random_seed"] == RANDOM_SEED)
        ALL_VERIFIED = ALL_VERIFIED and ok
        if not stats["all_finite"]:
            NONFINITE_FILES.append(path.name)
        verified_rows.append((v, temperature, top_k_label(top_k),
                              "OK" if ok else "FAILED", stats["abs_max"],
                              stats["n_nonfinite"]))

print(f"{'val':>3} {'temp':>5} {'top_k':>6}  {'status':<7} {'abs max':>10} {'non-finite':>11}")
for v, t, k, status, abs_max, n_nonfinite in verified_rows:
    shown = f"{abs_max:10.4f}" if abs_max is not None else " " * 10
    count = f"{n_nonfinite:11d}" if n_nonfinite is not None else " " * 11
    print(f"{v:>3} {t:>5.1f} {k:>6}  {status:<7} {shown} {count}")

print(f"\nAll {TOTAL_RUNS} files present and verified: {ALL_VERIFIED}")
if NONFINITE_FILES:
    print(f"Files containing non-finite values ({len(NONFINITE_FILES)}), "
          "saved unchanged and reported:")
    for name in NONFINITE_FILES:
        print(f"  {name}")
else:
    print("No non-finite values in any saved file.")

## 10. Inventory

Which configurations are complete and which still need a run. Use this after a
disconnect to see what is left, then re-run section 8's loop to resume.

In [ ]:
print("Chronos-T5 Base 200M - sampling-parameter validation")
print(f"  model       : {MODEL_ID} (loaded once, frozen, inference only)")
print(f"  variables   : temperature {TEMPERATURES}")
print(f"                top_k {[top_k_label(k) for k in TOP_K_VALUES]} "
      f"(full = {FULL_VOCABULARY_TOP_K})")
print(f"  fixed       : {NUM_SAMPLES} samples, top_p {TOP_P}, context {CONTEXT_LENGTH}, "
      f"horizon {PREDICTION_LENGTH}, seed {RANDOM_SEED}, batch {SERIES_BATCH_SIZE}")
print(f"  origins     : {len(list(VALIDATION_IDS))} pre-M6 validation contexts")
print()

complete = 0
print(f"  {'temperature':>11} {'top_k':>6}  {'saved':>7}  folder")
for temperature, top_k in CONFIGURATIONS:
    done = sum(1 for v in VALIDATION_IDS
               if samples_path(v, temperature, top_k).is_file())
    complete += done
    marker = "" if done == len(list(VALIDATION_IDS)) else "   <- incomplete"
    print(f"  {temperature:>11.1f} {top_k_label(top_k):>6}  "
          f"{done}/{len(list(VALIDATION_IDS))}    "
          f"{config_dir(temperature, top_k).name}{marker}")

print(f"\n  total saved : {complete}/{TOTAL_RUNS}")
print(f"  output base : {DRIVE_OUTPUT_BASE_DIR}")
print("  metrics     : none computed here (no RPS, ECE, confidence, entropy, shrinkage)")
if complete == TOTAL_RUNS:
    print("\n  Grid COMPLETE. Copy the configuration folders from Drive into")
    print("  TSFM Uncertainty Experiment/model_outputs/chronos_t5_base/validation/")
else:
    print("\n  Grid INCOMPLETE. Re-run the loop in section 8 to resume; "
          "finished files are skipped.")